# Mayavi viewer in a local notebook

Renders Mayavi output directly under the code cell, first as a static image and then as an interactive viewer you can rotate with the mouse.

Run it with the repo's `.venv` kernel (after `pip install -r requirements.txt`), in VS Code or in a browser via `jupyter lab`. Run the cells in order, top to bottom.

## Setup

In [ ]:
import asyncio

import numpy as np
from IPython.display import Image, display
from mayavi import mlab
from mayavi.tools import notebook
from mayavi.tools.remote.ipy_remote import IPyRemoteWidget
from mayavi.tools.remote.remote_scene import ImageEncoder
from tvtk.api import tvtk


def encode_frame(render_window, image_type="png", quality=60, render=True):
    """Return the window's current frame as PNG or JPEG bytes, rendering it first if asked."""
    if render:
        render_window.render()
    w2if = tvtk.WindowToImageFilter(input=render_window, read_front_buffer=False, should_rerender=False)
    if image_type == "png":
        writer = tvtk.PNGWriter(write_to_memory=True)
    else:
        writer = tvtk.JPEGWriter(quality=quality, write_to_memory=True)
    writer.set_input_connection(w2if.output_port)
    writer.write()
    return writer.result.to_array().tobytes()


# Fix 1: Mayavi 4.9.0's frame grabber calls ndarray.tostring() (removed in NumPy 2), and its cached
# pipeline can send a stale frame on mouse release, so the view snaps back. Grab every frame fresh.
# Mayavi calls this right after each render, so reuse that render instead of rendering again.
ImageEncoder.get_raw_image = lambda self: encode_frame(self.scene.render_window, self.image_type, self.quality, render=False)

# Fix 2: Mayavi passes the browser's key name (e.g. "Shift") to VTK as a one-character key code,
# which raises ValueError. Skip named keys. The original is stored once, so re-running this cell is safe.
if not hasattr(IPyRemoteWidget, "_original_handle_ipyevent"):
    IPyRemoteWidget._original_handle_ipyevent = IPyRemoteWidget.handle_ipyevent

def _skip_named_keys(self, event):
    if event["type"] in ("keydown", "keyup") and len(event.get("key", "")) != 1:
        return
    self._original_handle_ipyevent(event)

IPyRemoteWidget.handle_ipyevent = _skip_named_keys

mlab.init_notebook("ipy")  # also switches Mayavi to offscreen rendering

# Fix 3: Mayavi never schedules the throttled last frame of a drag. Run it on the kernel's event loop.
notebook._backend._widget_manager.sm.call_later = (
    lambda secs, f, *args, **kw: asyncio.get_event_loop().call_later(secs, lambda: f(*args, **kw))
)

## A. Static render

An image of the scene should appear below the cell.

In [ ]:
x, y = np.mgrid[-3:3:150j, -3:3:150j]
z = np.sin(x**2 + y**2) * np.exp(-0.2 * (x**2 + y**2))

fig = mlab.figure(bgcolor=(1, 1, 1), size=(600, 450))
mlab.surf(x, y, z, colormap="viridis", warp_scale=2)
display(Image(data=encode_frame(fig.scene.render_window)))
mlab.close(fig)

## B. Interactive viewer

Drag to rotate, scroll to zoom. Each frame is rendered by the Python kernel, so this needs the kernel running.

In [ ]:
fig = mlab.figure(bgcolor=(1, 1, 1), size=(600, 450))
mlab.surf(x, y, z, colormap="viridis", warp_scale=2)
mlab.view(azimuth=45, elevation=60, distance="auto")
fig  # displays the interactive viewer

## C. Self-check

In [ ]:
# Simulate a mouse drag on the viewer and confirm a new frame arrives.
viewer = notebook._backend._widget_manager.scene_to_ipy(fig.scene)
before = bytes(viewer.image.value)
assert len(before) > 1000, "viewer has no image"

for key in ("Shift", "Meta", "ArrowLeft"):  # named keys must not raise
    viewer.handle_ipyevent({"type": "keydown", "key": key})
    viewer.handle_ipyevent({"type": "keyup", "key": key})

camera_before = tuple(fig.scene.camera.position)
viewer.handle_ipyevent({"type": "mousedown", "button": 0, "relativeX": 300, "relativeY": 225})
for i in range(1, 31):  # Mayavi only acts on every third mouse move
    viewer.handle_ipyevent({"type": "mousemove", "button": 0, "relativeX": 300 + 4 * i, "relativeY": 225 + i})
viewer.handle_ipyevent({"type": "mouseup", "relativeX": 420, "relativeY": 255})
await asyncio.sleep(1)  # let the frame sent on release flush
assert tuple(fig.scene.camera.position) != camera_before, "drag did not rotate the camera"
assert viewer.image.format == "PNG", "final frame after release should be PNG"
assert bytes(viewer.image.value) != before, "view snapped back after the mouse was released"
print("Interactive viewer OK")